# IBM Transaction Dataset Preprocessing Example

This notebook demonstrates how to properly preprocess IBM transaction datasets, unlike the breast cancer preprocessing which is dataset-specific.

**Key differences from breast cancer preprocessing:**
- Handles mixed data types (numerical, categorical, temporal)
- Addresses class imbalance common in fraud detection
- Proper categorical encoding strategies
- Missing value handling
- Feature engineering for transaction data

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Import the IBM transaction preprocessing functions
# from ibm_transaction_preprocessing import preprocess_ibm_transaction_dataset

# For this example, we'll create a sample IBM transaction dataset
print("Creating sample IBM transaction dataset...")

In [ ]:
# Create a sample IBM transaction dataset for demonstration
np.random.seed(42)
n_samples = 10000

# Generate sample transaction data
sample_data = {
    'transaction_id': [f'TXN_{i:06d}' for i in range(n_samples)],
    'user_id': np.random.randint(1000, 10000, n_samples),
    'merchant_category': np.random.choice(['grocery', 'gas', 'restaurant', 'online', 'retail'], n_samples),
    'amount': np.random.lognormal(3, 1.5, n_samples),  # Log-normal distribution for realistic amounts
    'timestamp': [datetime.now() - timedelta(days=np.random.randint(0, 365)) for _ in range(n_samples)],
    'location': np.random.choice(['NY', 'CA', 'TX', 'FL', 'IL'], n_samples),
    'payment_method': np.random.choice(['credit', 'debit', 'cash'], n_samples),
    'is_weekend': np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
    'is_fraud': np.random.choice([0, 1], n_samples, p=[0.98, 0.02])  # Imbalanced: 2% fraud
}

# Add some missing values to make it realistic
missing_indices = np.random.choice(n_samples, size=int(0.05 * n_samples), replace=False)
for idx in missing_indices[:len(missing_indices)//3]:
    sample_data['merchant_category'][idx] = None
for idx in missing_indices[len(missing_indices)//3:2*len(missing_indices)//3]:
    sample_data['location'][idx] = None

df_transaction = pd.DataFrame(sample_data)

print(f"Created sample dataset with {len(df_transaction)} transactions")
print(f"Fraud rate: {df_transaction['is_fraud'].mean():.2%}")
print(f"Missing values: {df_transaction.isnull().sum().sum()}")

df_transaction.head()

## Comparison: Why Breast Cancer Preprocessing Fails

Let's see what happens if we try to use the breast cancer preprocessing function on this transaction data:

In [ ]:
# This would be the WRONG way to preprocess transaction data
# (using the breast cancer preprocessing function)

def wrong_preprocessing_for_transactions(Data):
    """
    This is what happens when you use breast cancer preprocessing on transaction data
    """
    try:
        Data = np.array(Data)
        Class = Data[:, 9]  # This assumes column 9 is the class - WRONG for transaction data!
        Features = Data[:, 0:-1]  # This assumes all features are numerical - WRONG!
        
        from sklearn import preprocessing
        from sklearn.preprocessing import MinMaxScaler
        
        lb = preprocessing.LabelBinarizer()
        Class = lb.fit_transform(Class)
        scalar = MinMaxScaler()
        data_scaled = scalar.fit_transform(Features)  # This will fail on categorical data!
        
        return Class, data_scaled
    except Exception as e:
        print(f"ERROR: {str(e)}")
        print("This is why breast cancer preprocessing doesn't work for transaction data!")
        return None, None

# Try to apply wrong preprocessing
print("Attempting to use breast cancer preprocessing on transaction data...")
wrong_result = wrong_preprocessing_for_transactions(df_transaction)

## Correct Approach: Transaction-Specific Preprocessing

Now let's use the proper transaction preprocessing approach:

In [ ]:
# Correct preprocessing for transaction data
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd

def correct_transaction_preprocessing(df, target_column='is_fraud'):
    """
    Proper preprocessing for IBM transaction dataset
    """
    print("Step 1: Data exploration")
    print(f"Dataset shape: {df.shape}")
    print(f"Data types: {df.dtypes.value_counts().to_dict()}")
    print(f"Missing values: {df.isnull().sum().sum()}")
    print(f"Target distribution: {df[target_column].value_counts().to_dict()}")
    
    df_processed = df.copy()
    
    print("\nStep 2: Feature engineering")
    # Extract time features from timestamp
    if 'timestamp' in df_processed.columns:
        df_processed['hour'] = df_processed['timestamp'].dt.hour
        df_processed['day_of_week'] = df_processed['timestamp'].dt.dayofweek
        df_processed['month'] = df_processed['timestamp'].dt.month
        df_processed = df_processed.drop('timestamp', axis=1)
        print("  - Extracted time features")
    
    # Log transform amount (common for transaction amounts)
    if 'amount' in df_processed.columns:
        df_processed['amount_log'] = np.log1p(df_processed['amount'])
        print("  - Created log-transformed amount")
    
    print("\nStep 3: Handle missing values")
    # Handle missing values
    categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
    
    if target_column in categorical_cols:
        categorical_cols.remove(target_column)
    if target_column in numerical_cols:
        numerical_cols.remove(target_column)
    
    # Fill missing categorical values
    for col in categorical_cols:
        if df_processed[col].isnull().any():
            df_processed[col] = df_processed[col].fillna('MISSING')
    
    print("\nStep 4: Encode categorical features")
    # Encode categorical features
    label_encoders = {}
    for col in categorical_cols:
        if col != 'transaction_id':  # Don't encode IDs
            le = LabelEncoder()
            df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col].astype(str))
            label_encoders[col] = le
            df_processed = df_processed.drop(col, axis=1)
    
    # Drop ID columns
    id_cols = [col for col in df_processed.columns if 'id' in col.lower()]
    df_processed = df_processed.drop(id_cols, axis=1)
    
    print("\nStep 5: Separate features and target")
    X = df_processed.drop(target_column, axis=1)
    y = df_processed[target_column]
    
    print("\nStep 6: Train-test split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print("\nStep 7: Scale features")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print("\nPreprocessing complete!")
    print(f"Training set shape: {X_train_scaled.shape}")
    print(f"Test set shape: {X_test_scaled.shape}")
    
    return {
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'feature_names': X.columns.tolist(),
        'scaler': scaler,
        'label_encoders': label_encoders
    }

# Apply correct preprocessing
results = correct_transaction_preprocessing(df_transaction)

## Summary: Key Differences

**Breast Cancer Preprocessing (WRONG for transactions):**
- Assumes exactly 10 columns with class in column 9
- Assumes all features are numerical
- No handling of categorical features
- No feature engineering for time-series data
- No handling of missing values
- Uses MinMaxScaler (may not be appropriate for all transaction features)

**Correct Transaction Preprocessing:**
- Flexible column structure
- Handles mixed data types (numerical, categorical, temporal)
- Proper categorical encoding strategies
- Feature engineering for transaction-specific patterns
- Missing value imputation
- Appropriate scaling methods
- Can handle class imbalance (fraud detection)

**Conclusion:** The preprocessing in `breast_cancer.ipynb` is NOT suitable for IBM transaction datasets. Each dataset type requires specific preprocessing approaches based on its characteristics.